# nanoCalc-GPT Evaluation
Evaluate the saved model checkpoint on train and test datasets using exact match accuracy and per-operator breakdowns.

In [9]:
from exact_match import load_saved_model, exact_match, exact_match_by_operator, operator_accuracy, predict
from model import GPT, GPTConfig

# Load saved model from artifacts
model, config = load_saved_model("artifacts/model.pt")
print(f"Loaded model successfully on device: {config.device}")


Loaded model successfully on device: cpu


In [10]:
# Evaluate exact match accuracy on Test set with formatted operator table
test_results = exact_match(model, data="test")
print(test_results.summary(show_table=True))

=== Exact Match | Split: test ===
Overall Accuracy: 2982/3057 (97.5466%)

+--------------------+---------+---------+--------+----------+
| Operator           |   Total | Correct | Errors | Accuracy |
+--------------------+---------+---------+--------+----------+
| * (Multiplication) |    1000 |     939 |     61 |  93.90% |
| + (Addition)       |    1000 |    1000 |      0 | 100.00% |
| - (Subtraction)    |    1000 |    1000 |      0 | 100.00% |
| / (Division)       |      57 |      43 |     14 |  75.44% |
+--------------------+---------+---------+--------+----------+
| Overall            |    3057 |    2982 |     75 |  97.55% |
+--------------------+---------+---------+--------+----------+


In [11]:
# Per-operator accuracy queries and metrics
for op in test_results.operators:
    name = test_results.by_operator[op]['name']
    acc = test_results.op_accuracy(op)
    corr = test_results.op_correct(op)
    tot = test_results.op_total(op)
    errs = test_results.op_errors(op)
    print(f"{op} ({name}): {corr}/{tot} = {acc:.2%} (Errors: {errs})")

# Inspect error cases for division
print("\n--- Division Error Samples ---")
for s in test_results.get_errors('/'):
    print(f"{s['prompt']} -> Pred: {s['predicted']} (decoded: {s['decoded_pred']}) | Expected: {s['target']} (expected: {s['decoded_target']})")


* (Multiplication): 939/1000 = 93.90% (Errors: 61)
+ (Addition): 1000/1000 = 100.00% (Errors: 0)
- (Subtraction): 1000/1000 = 100.00% (Errors: 0)
/ (Division): 43/57 = 75.44% (Errors: 14)

--- Division Error Samples ---
84/06= -> Pred: +0012 (decoded: 12) | Expected: +0014 (expected: 14)
88/04= -> Pred: +0021 (decoded: 21) | Expected: +0022 (expected: 22)
24/04= -> Pred: +0004 (decoded: 4) | Expected: +0006 (expected: 6)
52/04= -> Pred: +0017 (decoded: 17) | Expected: +0013 (expected: 13)
48/08= -> Pred: +0005 (decoded: 5) | Expected: +0006 (expected: 6)
32/02= -> Pred: +0011 (decoded: 11) | Expected: +0016 (expected: 16)
84/03= -> Pred: +0027 (decoded: 27) | Expected: +0028 (expected: 28)
18/02= -> Pred: +0008 (decoded: 8) | Expected: +0009 (expected: 9)
60/04= -> Pred: +0010 (decoded: 10) | Expected: +0015 (expected: 15)
80/08= -> Pred: +0006 (decoded: 6) | Expected: +0010 (expected: 10)


In [4]:
# Filter evaluation to a single operator directly
mult_only = exact_match(model, data="test", operator="*")
print(mult_only.summary(show_table=False))

=== Exact Match | Split: test | Filter: * ===
Overall Accuracy: 939/1000 (93.9000%)
Per-Operator Accuracy:
  Op '*' (Multiplication): 939/1000 (93.9000%)


In [5]:
# Evaluate exact match accuracy on Train set
train_results = exact_match(model, data="train")
print(train_results.summary(show_table=True))

# Train set evaluation takes ~2-3 min on CPU.
# Run this cell to evaluate; interrupt with Kernel > Interrupt if needed.


In [15]:
# Interactive predictions
prompts = ["12+34=", "85-23=", "15*06=", "72/08=", "99*99="]
for p in prompts:
    res = predict(model, p)
    print(f"{res['prompt']} -> Raw: {res['raw_prediction']} | Decoded: {res['math_value']}")

12+34= -> Raw: +6400 | Decoded: 46
85-23= -> Raw: +2600 | Decoded: 62
15*06= -> Raw: +0900 | Decoded: 90
72/08= -> Raw: +0009 | Decoded: 9
99*99= -> Raw: +1089 | Decoded: 9801
